# 04c · Audit of large transfers (Mt. Gox, governments, custodians)

Inputs: tables from 04 and 04b

Outputs: `large_movers.csv`, `large_movers_by_day.csv`; after manual labeling, additional rows in `custodial_keys`

Workflow: list candidate key-days, identify entities by hand, merge confirmed institutional keys into `custodial_keys`.

1. List (step 1): key-days in the tagged windows on which one key sent (y3) ≥ `MIN_BTC` (default 1,000 BTC), excluding keys already flagged as services, exchanges or custodians. Each row has a representative address (the key's address with the largest amount spent that day), the largest transaction, the destination split and the event or placebo windows it falls in.
2. Merge (step 2): `large_movers_labeled.csv` adds `entity`, `entity_type` (mtgox / government / custodian / exchange / fund / individual / unknown), `source_url` (primary sources preferred) and `exclude`. Keys with `exclude = TRUE` are added to `custodial_keys` (src = `audit:<entity>`), and `key_flags` is updated as in 04b (idempotent). The labeled file is published as Data S2.

Exclusion rule: only movements by institutions or under legal proceedings (exchanges, custodians, bankruptcy trustees, governments, fund/ETF custody). Unidentified large holders (entity_type = individual / unknown) stay in the sample and are reported separately in the BTC-weighted robustness checks.

In [ ]:
# Configuration
PROJECT_ID = "your-gcp-project"
DATASET    = "qbtc_raw"
LOCATION   = "US"

AUTO_RUN_USD      = 5.0
MAX_USD_PER_QUERY = 10.0
MAX_USD_TOTAL     = 100.0
PRICE_PER_TIB_USD = 6.25
CONFIRM           = False
FORCE_REBUILD     = False

RISK_MIN_BTC       = 0.01    # risk set: minimum balance at t-1
RISK_LOOKBACK_DAYS = 365     # risk set: incoming transfer within 12 months (not applied to positive controls)
SERVICE_MAX_RECV   = 1000    # any address with more lifetime receipts -> service/custodian, excluded
MAIN_WIN   = (-14, 30)       # main window
WIDE_WIN   = (-60, 60)       # wider window for main events and positive controls, for pre-trend plots
DORMANT_DAYS = 30            # y1 strict: unspent for the next 30 days
EXPAND_COSPEND   = True      # expand exchange labels by one hop of common-input spending
COSPEND_MAX_IN   = 1000      # expansion uses only transactions with at most this many inputs
N_DRAWS, SEED    = 1000, 20260923
AUDIT_N          = 200       # size of the y1-strict audit sample

try:
    from google.colab import auth
    auth.authenticate_user()
except ImportError:
    pass

from google.cloud import bigquery
import json, os, io, re, time, zipfile, datetime as dt
import numpy as np
import pandas as pd
import requests

client = bigquery.Client(project=PROJECT_ID, location=LOCATION)
DS = f"{PROJECT_ID}.{DATASET}"

LEDGER_PATH, JOBS_PATH = "build_ledger.json", "jobs.json"
LEDGER = json.load(open(LEDGER_PATH)) if os.path.exists(LEDGER_PATH) else {"jobs": [], "usd_total": 0.0}
JOBS   = json.load(open(JOBS_PATH))   if os.path.exists(JOBS_PATH)   else []

for t in ["flow_key_tx", "key_flags", "events", "event_flow_coverage", "placebo_days", "placebo_draws_strict", "custodial_keys", "flow_tx_inputs", "flow_days"]:
    client.get_table(f"{DS}.{t}")        # raises if a table is missing
print("Tables from 04 and 04b found. Spent in this Colab session: $%.2f / $%.0f (the ledger file lives in the current session only)" % (LEDGER["usd_total"], MAX_USD_TOTAL))

In [ ]:
# Helpers
def usd(gb): return gb / 1024 * PRICE_PER_TIB_USD

def dry_run(sql):
    job = client.query(sql, job_config=bigquery.QueryJobConfig(dry_run=True, use_query_cache=False))
    gb = job.total_bytes_processed / 2**30
    return gb, usd(gb)

def table_rows(name):
    try:
        return client.get_table(f"{DS}.{name}").num_rows
    except Exception:
        return None

def table_info(name):
    t = client.get_table(f"{DS}.{name}")
    return {"table": name, "rows": t.num_rows, "GB": round(t.num_bytes / 2**30, 2)}

def build(name, sql, force=False):
    assert not re.search(r"CLUSTER BY[^\n]*\b(key_id|txid|spending_txid)\b", sql), f"[{name}] cannot cluster on a BYTES column"
    n = table_rows(name)
    if n and not (FORCE_REBUILD or force):
        print(f"[{name}] exists, {n:,} rows; skipped (rebuild with build(..., force=True))")
        return True
    gb, cost = dry_run(sql)
    if cost > MAX_USD_PER_QUERY:
        raise RuntimeError(f"[{name}] estimated ${cost:.2f}, above the per-query limit ${MAX_USD_PER_QUERY}")
    if LEDGER["usd_total"] + cost > MAX_USD_TOTAL:
        raise RuntimeError(f"[{name}] total would exceed ${MAX_USD_TOTAL}")
    if cost > AUTO_RUN_USD and not CONFIRM:
        raise RuntimeError(f"[{name}] estimated {gb:,.0f} GB ~ ${cost:.2f}, above the auto-run limit ${AUTO_RUN_USD}: set CONFIRM=True and re-run this cell (tables already built are skipped)")
    print(f"[{name}] estimated {gb:,.0f} GB ~ ${cost:.2f}")
    job = client.query(sql, job_config=bigquery.QueryJobConfig(
        maximum_bytes_billed=int(gb * 1.25 * 2**30) + 10 * 2**30, use_query_cache=False))
    JOBS.append({"table": name, "job_id": job.job_id, "location": LOCATION,
                 "submitted_at": dt.datetime.now(dt.timezone.utc).isoformat()})
    json.dump(JOBS, open(JOBS_PATH, "w"), indent=2)
    print(f"[{name}] submitted job {job.job_id} (runs on the server; finishes even if Colab disconnects)")
    t0 = time.time()
    job.result()
    billed = (job.total_bytes_billed or 0) / 2**30
    LEDGER["jobs"].append({"table": name, "job_id": job.job_id, "billed_gb": round(billed, 1),
                           "usd": round(usd(billed), 2), "seconds": round(time.time() - t0, 1)})
    LEDGER["usd_total"] = round(LEDGER["usd_total"] + usd(billed), 2)
    json.dump(LEDGER, open(LEDGER_PATH, "w"), indent=2)
    print(f"[{name}] done, {billed:,.0f} GB, {LEDGER['jobs'][-1]['seconds']}s", table_info(name))
    return True

def need(*names):
    missing = [n for n in names if not table_rows(n)]
    assert not missing, f"These tables are not built yet: {missing} (run the earlier cells first)"

def q(sql):
    return client.query(sql).to_dataframe()

def upload(name, df):
    df = df.copy()
    for c in df.columns:                       # dates are always stored as DATE (joinable with the date columns of utxo_life)
        if pd.api.types.is_datetime64_any_dtype(df[c]):
            df[c] = df[c].dt.date
    job = client.load_table_from_dataframe(
        df, f"{DS}.{name}", job_config=bigquery.LoadJobConfig(write_disposition="WRITE_TRUNCATE"))
    job.result()
    print(f"[{name}] uploaded {len(df):,} rows")

SATS_MIN = int(round(RISK_MIN_BTC * 1e8))
FREEZE_DATE = "2026-09-22"   # freeze date = latest snapshot date (utxo_life has only part of the blocks of 09-23, which are not used)
DATA_END = pd.Timestamp(FREEZE_DATE)
_latest = pd.Timestamp(q(f"SELECT MAX(snap_date) AS d FROM `{DS}.holder_snapshot` WHERE event = 'latest'").d.iloc[0])
assert _latest == DATA_END, f"latest snapshot date {_latest.date()} differs from the freeze date {FREEZE_DATE}"
print("Freeze date (last data day):", DATA_END.date())

## Step 1 · List

In [ ]:
MIN_BTC = 1000
SQL_LM = f'''
CREATE OR REPLACE TABLE `{DS}.large_movers` AS
WITH kd AS (
  SELECT f.key_id, f.date, f.n_tx, f.spent_sats, f.y3, f.y2, f.y0, f.y1_loose, f.y1_strict, f.change_sats
  FROM `{DS}.key_day_flows` f
  JOIN `{DS}.key_flags` k USING (key_id)
  WHERE NOT k.is_service AND f.y3 >= {int(MIN_BTC * 1e8)}
),
win AS (
  SELECT date, STRING_AGG(DISTINCT CONCAT(event, '(', CAST(rel_day AS STRING), ')'), ' ' ORDER BY CONCAT(event, '(', CAST(rel_day AS STRING), ')')) AS windows,
         LOGICAL_OR(event_set = 'main' AND in_main_window) AS in_main_window
  FROM `{DS}.flow_days` GROUP BY date
),
addr AS (
  SELECT i.key_id, i.spent_date AS date,
         ARRAY_AGG(i.address ORDER BY i.sats DESC LIMIT 1)[OFFSET(0)] AS rep_address,
         COUNT(DISTINCT i.address) AS n_addr_spent
  FROM `{DS}.flow_tx_inputs` i
  JOIN kd ON kd.key_id = i.key_id AND kd.date = i.spent_date
  WHERE i.is_risk_key
  GROUP BY 1, 2
),
toptx AS (
  SELECT t.key_id, t.date, ARRAY_AGG(TO_HEX(t.txid) ORDER BY t.y3 DESC LIMIT 1)[OFFSET(0)] AS top_txid
  FROM `{DS}.flow_key_tx` t
  JOIN kd USING (key_id, date)
  GROUP BY 1, 2
)
SELECT kd.date, TO_HEX(kd.key_id) AS key_id_hex, a.rep_address, a.n_addr_spent,
       CONCAT('https://mempool.space/address/', a.rep_address) AS address_url,
       CONCAT('https://mempool.space/tx/', tt.top_txid)        AS top_tx_url,
       kd.n_tx,
       ROUND(kd.spent_sats / 1e8, 2) AS spent_btc, ROUND(kd.y3 / 1e8, 2) AS y3_btc,
       ROUND(kd.y1_strict / 1e8, 2) AS y1_strict_btc, ROUND(kd.y1_loose / 1e8, 2) AS y1_loose_btc,
       ROUND(kd.y0 / 1e8, 2) AS y0_btc, ROUND(kd.y2 / 1e8, 2) AS y2_btc,
       k.first_seen_date, k.max_addr_n_received, k.in_main, k.in_placebo, k.in_pc,
       w.in_main_window, w.windows
FROM kd
JOIN `{DS}.key_flags` k USING (key_id)
LEFT JOIN addr a USING (key_id, date)
LEFT JOIN toptx tt USING (key_id, date)
LEFT JOIN win w USING (date)
'''
build("large_movers", SQL_LM, force=True)
lm = q(f"SELECT * FROM `{DS}.large_movers` ORDER BY date, y3_btc DESC")
lm["date"] = pd.to_datetime(lm.date)
for c in ["entity", "entity_type", "source_url", "exclude"]:
    lm[c] = ""
print(f"{len(lm)} key-days ({lm.key_id_hex.nunique()} keys), total y3 {lm.y3_btc.sum():,.0f} BTC; "
      f"{int(lm.in_main_window.fillna(False).sum())} inside main-event main windows")
by_day = (lm.groupby("date").agg(n_keys=("key_id_hex", "nunique"), y3_btc=("y3_btc", "sum"),
                                 y1_strict_btc=("y1_strict_btc", "sum"), y0_btc=("y0_btc", "sum"),
                                 windows=("windows", "first"))
            .sort_values("y3_btc", ascending=False).round(0))
display(by_day.head(30))
lm.to_csv("large_movers.csv", index=False)
by_day.to_csv("large_movers_by_day.csv")
try:
    from google.colab import files
    files.download("large_movers.csv"); files.download("large_movers_by_day.csv")
except Exception:
    print("Saved large_movers.csv and large_movers_by_day.csv")

## Step 2 · Merge into `custodial_keys`

Requires `large_movers_labeled.csv` in the working directory. `exclude` accepts TRUE/True/1/yes.

In [ ]:
lab = pd.read_csv("large_movers_labeled.csv", dtype=str).fillna("")
lab["exclude"] = lab.exclude.str.strip().str.lower().isin(["true", "1", "yes", "y"])
bad = lab[lab.exclude & (lab.entity.str.strip() == "")]
assert bad.empty, f"exclude=TRUE without an entity: {len(bad)} rows"
ex = (lab[lab.exclude][["key_id_hex", "entity", "entity_type", "source_url"]]
        .drop_duplicates("key_id_hex").reset_index(drop=True))
print(f"Marked for exclusion: {len(ex)} keys; by type:", ex.entity_type.value_counts().to_dict())
display(lab.groupby(["entity_type", "exclude"]).agg(n=("key_id_hex", "nunique"),
        y3_btc=("y3_btc", lambda s: pd.to_numeric(s).sum())).round(0))
upload("large_movers_labeled", lab)

# custodial_keys: keep the Coinbase rows from 04b, replace earlier audit rows
SQL_CK = f'''
CREATE OR REPLACE TABLE `{DS}.custodial_keys` AS
SELECT key_id, src, n_tx, sats, first_date FROM `{DS}.custodial_keys` WHERE NOT STARTS_WITH(src, 'audit:')
UNION ALL
SELECT FROM_HEX(l.key_id_hex) AS key_id, CONCAT('audit:', ANY_VALUE(l.entity)) AS src,
       CAST(NULL AS INT64) AS n_tx, CAST(NULL AS INT64) AS sats, CAST(NULL AS DATE) AS first_date
FROM `{DS}.large_movers_labeled` l
WHERE l.exclude
  AND FROM_HEX(l.key_id_hex) NOT IN (SELECT key_id FROM `{DS}.custodial_keys` WHERE NOT STARTS_WITH(src, 'audit:'))
GROUP BY l.key_id_hex
'''
build("custodial_keys", SQL_CK, force=True)

names = [f.name for f in client.get_table(f"{DS}.key_flags").schema]
assert "is_service_base" in names, "run 04b first"
build("key_flags", f'''
CREATE OR REPLACE TABLE `{DS}.key_flags` AS
SELECT kf.* EXCEPT(is_service_base, is_service, is_custodial),
       kf.is_service_base,
       c.key_id IS NOT NULL                      AS is_custodial,
       (kf.is_service_base OR c.key_id IS NOT NULL) AS is_service
FROM `{DS}.key_flags` kf
LEFT JOIN (SELECT DISTINCT key_id FROM `{DS}.custodial_keys`) c USING (key_id)
''', force=True)
summ = q(f'''
SELECT c.src, COUNT(*) AS n_keys, COUNTIF(k.in_main) AS in_main
FROM `{DS}.custodial_keys` c LEFT JOIN `{DS}.key_flags` k USING (key_id)
GROUP BY 1 ORDER BY 2 DESC''')
display(summ)
report = {"generated_at": dt.datetime.now(dt.timezone.utc).isoformat(), "min_btc": MIN_BTC,
          "n_rows": len(lab), "n_excluded_keys": len(ex),
          "excluded_by_type": ex.entity_type.value_counts().to_dict(),
          "custodial_keys": summ.astype(str).to_dict(orient="records"), "ledger": LEDGER}
with open("build_report_04c.json", "w", encoding="utf-8") as f:
    json.dump(report, f, ensure_ascii=False, indent=2, default=str)
try:
    from google.colab import files
    files.download("build_report_04c.json")
except Exception:
    print("Report saved to build_report_04c.json")